In [2]:
import os


In [3]:
%pwd

'd:\\Text-Summerizer-project\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'd:\\Text-Summerizer-project'

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path


In [7]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [ ]:
class  ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self):

        config = self.config.model_evaluation

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,
            tokenizer_path=config.tokenizer_path,
            metric_file_name=config.metric_file_name
        )

        return model_evaluation_config
        
        

In [9]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk, load_metric
import torch
import pandas as pd
from tqdm import tqdm

d:\Text-Summerizer-project\textS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-06-05 16:13:22,862:INFO:datasets:PyTorch version 2.11.0+cu128 available.]


In [10]:
import os

print(os.getcwd())

d:\Text-Summerizer-project


In [11]:
import os

os.chdir(r"D:\Text-Summerizer-project")
print(os.getcwd())

D:\Text-Summerizer-project


In [12]:
from datasets import load_from_disk

dataset_samsum = load_from_disk(
    "artifacts/data_transformation/samsum_dataset"
)

In [13]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

In [23]:
from transformers import AutoModelForSeq2SeqLM

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

In [24]:
import evaluate

metric = evaluate.load("rouge")

In [25]:
from datasets import load_from_disk

dataset_samsum = load_from_disk(
    r"D:\Text-Summerizer-project\artifacts\data_transformation\samsum_dataset"
)

In [31]:
import gc
import torch
from tqdm import tqdm
import evaluate
import pandas as pd
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


class ModelEvaluation:
    def __init__(self, config):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i:i + batch_size]

    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=1,
        device="cpu",
        column_text="dialogue",
        column_summary="summary"
    ):

        model.eval()

        article_batches = list(
            self.generate_batch_sized_chunks(dataset[column_text], batch_size)
        )

        target_batches = list(
            self.generate_batch_sized_chunks(dataset[column_summary], batch_size)
        )

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches)
        ):

            inputs = tokenizer(
                article_batch,
                max_length=128,
                truncation=True,
                padding=True,
                return_tensors="pt"
            )

            # ✅ FIX: ensure SAME device as model
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            with torch.no_grad():
                summaries = model.generate(
                    **inputs,
                    max_length=32,
                    num_beams=1,
                    early_stopping=True
                )

            decoded_summaries = tokenizer.batch_decode(
                summaries,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )

            metric.add_batch(
                predictions=decoded_summaries,
                references=target_batch
            )

            del inputs
            del summaries
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return metric.compute()

    def evaluate(self):

        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)

        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path
        ).to(device)   # ✅ FIX: model moved to GPU

        dataset_samsum = load_from_disk(self.config.data_path)

        rouge_metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(
            dataset=dataset_samsum["test"].select(range(10)),  # small test for speed
            metric=rouge_metric,
            model=model,
            tokenizer=tokenizer,
            batch_size=2,
            device=device,
            column_text="dialogue",
            column_summary="summary"
        )

        print(score)

        pd.DataFrame([score]).to_csv(
            self.config.metric_file_name,
            index=False
        )

        return score

In [32]:
config = ConfigurationManager()

model_eval_config = config.get_model_evaluation_config()

print(model_eval_config)
print(type(model_eval_config))

[2026-06-05 16:19:50,265:INFO:textSummarizerLogger:yaml file: config\config.yaml loaded successfully]
[2026-06-05 16:19:50,268:INFO:textSummarizerLogger:yaml file: params.yaml loaded successfully]
[2026-06-05 16:19:50,270:INFO:textSummarizerLogger:created directory at: artifacts]
ModelEvaluationConfig(root_dir='artifacts/model_evaluation', data_path='artifacts/data_transformation/samsum_dataset', model_path='artifacts/model_trainer/flan-t5-small-samsum-model', tokenizer_path='artifacts/model_trainer/tokenizer', metric_file_name='artifacts/model_evaluation/metrics.csv')
<class '__main__.ModelEvaluationConfig'>


In [28]:
import os
print(os.getcwd())

D:\Text-Summerizer-project


In [29]:
import os
os.chdir(r"D:\Text-Summerizer-project")
print(os.getcwd())

D:\Text-Summerizer-project


In [33]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2026-06-05 16:19:53,750:INFO:textSummarizerLogger:yaml file: config\config.yaml loaded successfully]
[2026-06-05 16:19:53,754:INFO:textSummarizerLogger:yaml file: params.yaml loaded successfully]
[2026-06-05 16:19:53,756:INFO:textSummarizerLogger:created directory at: artifacts]


100%|██████████| 5/5 [00:07<00:00,  1.54s/it]

[2026-06-05 16:20:04,693:INFO:absl:Using default tokenizer.]


{'rouge1': np.float64(0.3798775759814084), 'rouge2': np.float64(0.13003619723514137), 'rougeL': np.float64(0.30161432396738275), 'rougeLsum': np.float64(0.30236169067924046)}


In [34]:
from pathlib import Path

root_dir = Path("D:/Text-Summerizer-project/artifacts/data_transformation")